# 07 — Path patching fiel ao artigo

Demonstramos duas etapas do circuito publicado: **Duplicate/Induction→S-Inhibition** via V em `S2` e **S-Inhibition→Name Movers** via Q em `END`.

`paper_path_patch` executa os quatro forwards históricos: novo para cachear sender, original para cachear Q/K/V, original com sender novo e os demais inputs congelados para capturar receiver, e original com somente o receiver capturado. MLPs são recomputados por padrão, como no artigo e no `ioi_utils.path_patching` histórico.

In [1]:
import json
from pathlib import Path

import plotly.express as px
import torch
from transformer_lens import patching as tl_patching

from ioi_replication.paper import (
    HeadPositionNode,
    PathReceiver,
    SemanticPosition,
    build_paired_dataset,
    logit_difference,
    paper_path_patch,
)
from ioi_replication.runtime import load_gpt2_small, select_device

SEED = 107
N_PATH = 2
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULT_PATH = PROJECT_ROOT / "results/paper/07_path_patching.json"

device = select_device()
model = load_gpt2_small(device)
dataset = build_paired_dataset(model.tokenizer, n=N_PATH, seed=SEED)
ioi_tokens = dataset.ioi_tokens.to(device)
abc_tokens = dataset.abc_tokens.to(device)

def metric(logits):
    return logit_difference(
        logits,
        dataset.io_token_ids.to(device),
        dataset.s_token_ids.to(device),
        dataset.positions["END"].to(device),
    )

print(f"device={device}; N={N_PATH}; baseline IO−S={metric(model(ioi_tokens)).item():.4f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

device=cuda; N=2; baseline IO−S=3.0995


## Intervenções históricas em posições semânticas

Cada sender e receiver abaixo traz sua posição explícita. Assim, `S2` e `END` são reunidos separadamente para cada linha do batch; nenhum hook substitui todas as posições.

In [2]:
interventions = {
    "duplicate_3.0_to_s_inhibition_7.3_v": (
        HeadPositionNode(3, 0, SemanticPosition.S2),
        PathReceiver(7, 3, "v", SemanticPosition.S2),
    ),
    "induction_5.5_to_s_inhibition_7.3_v": (
        HeadPositionNode(5, 5, SemanticPosition.S2),
        PathReceiver(7, 3, "v", SemanticPosition.S2),
    ),
    "s_inhibition_7.3_to_name_mover_9.9_q": (
        HeadPositionNode(7, 3, SemanticPosition.END),
        PathReceiver(9, 9, "q", SemanticPosition.END),
    ),
}

historical = {}
for name, (sender, receiver) in interventions.items():
    result = paper_path_patch(
        model,
        ioi_tokens,
        abc_tokens,
        dataset.positions,
        dataset.positions,
        senders=(sender,),
        receivers=(receiver,),
        metric=metric,
        freeze_mlps=False,
    )
    historical[name] = {
        "baseline": result.baseline.item(),
        "patched": result.patched.item(),
        "effect": result.effect.item(),
        "sender_position": sender.position.value,
        "receiver_position": receiver.position.value,
        "receiver_component": receiver.component.value,
    }
    print(
        f"{name}: {result.baseline.item():+.4f} → "
        f"{result.patched.item():+.4f} (Δ={result.effect.item():+.4f})"
    )

duplicate_3.0_to_s_inhibition_7.3_v: +3.0995 → +3.0107 (Δ=-0.0889)


induction_5.5_to_s_inhibition_7.3_v: +3.0995 → +2.9818 (Δ=-0.1178)


s_inhibition_7.3_to_name_mover_9.9_q: +3.0995 → +2.9979 (Δ=-0.1017)


## Diagnóstico moderno `get_act_patch_direct_path`

A comparação usa a mesma métrica bruta IO−S nos dois métodos e salva os endpoints separadamente. Não misturamos recuperação normalizada com diferença de logits.

O helper do TransformerLens 4.3 é uma intervenção diferente: soma o write `z @ W_O` novo−original no fork pré-Q/K/V de um receiver e o faz em **todas as posições**. Ele não executa os quatro passes históricos, não congela a rede intermediária do mesmo modo e não aceita as posições semânticas variáveis desta análise.

In [3]:
sender_hook_names = sorted({
    f"blocks.{sender.layer}.attn.hook_z"
    for sender, _ in interventions.values()
})
_, abc_z_cache = model.run_with_cache(abc_tokens, names_filter=sender_hook_names)
_, ioi_z_cache = model.run_with_cache(ioi_tokens, names_filter=sender_hook_names)

modern_direct = {}
for name, (sender, receiver) in interventions.items():
    matrix = tl_patching.get_act_patch_direct_path(
        model=model,
        corrupted_tokens=ioi_tokens,
        clean_cache=abc_z_cache,
        corrupted_cache=ioi_z_cache,
        patching_metric=metric,
        src_layer=sender.layer,
        src_head=sender.head,
        component=receiver.component.value,
        verbose=False,
    )
    patched = matrix[receiver.layer, receiver.head]
    baseline = torch.tensor(historical[name]["baseline"], device=patched.device)
    modern_direct[name] = {
        "baseline": baseline.item(),
        "patched": patched.item(),
        "effect": (patched - baseline).item(),
        "position_scope": "all_positions",
    }
    print(
        f"{name}: {baseline.item():+.4f} → {patched.item():+.4f} "
        f"(Δ={patched.item() - baseline.item():+.4f}; all-pos)"
    )

plot_rows = []
for name in interventions:
    plot_rows.extend([
        {"edge": name, "method": "paper four-pass / semantic", "effect": historical[name]["effect"]},
        {"edge": name, "method": "TL direct-path / all-pos", "effect": modern_direct[name]["effect"]},
    ])
fig = px.bar(
    plot_rows,
    x="edge",
    y="effect",
    color="method",
    barmode="group",
    title="Efeito bruto em IO−S: algoritmos distintos",
    labels={"effect": "Δ IO−S", "edge": "aresta"},
)
fig.update_layout(width=950, height=520, xaxis_tickangle=-20)
fig.show()

duplicate_3.0_to_s_inhibition_7.3_v: +3.0995 → +3.0022 (Δ=-0.0973; all-pos)


induction_5.5_to_s_inhibition_7.3_v: +3.0995 → +2.9343 (Δ=-0.1652; all-pos)


s_inhibition_7.3_to_name_mover_9.9_q: +3.0995 → +2.9863 (Δ=-0.1132; all-pos)


In [4]:
artifact = {
    "paper": "arXiv:2211.00593",
    "config": {
        "model_name": "gpt2",
        "seed": SEED,
        "n_examples": N_PATH,
        "dataset_sha256": dataset.dataset_sha256,
        "metric": "mean IO-minus-S logit difference at semantic END",
        "freeze_mlps": False,
    },
    "historical_four_pass_semantic": historical,
    "transformerlens_direct_path_all_positions": modern_direct,
    "comparison_warning": (
        "Both sections use raw IO-S logit difference, but the interventions "
        "are not semantically interchangeable."
    ),
}
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULT_PATH.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
print(f"Resultado salvo em {RESULT_PATH.relative_to(PROJECT_ROOT)}")
artifact["config"]

Resultado salvo em results\paper\07_path_patching.json


{'model_name': 'gpt2',
 'seed': 107,
 'n_examples': 2,
 'dataset_sha256': 'a9c668d845ab9fef7630428c87155240aef93f5033462576923204d5cb6dec8d',
 'metric': 'mean IO-minus-S logit difference at semantic END',
 'freeze_mlps': False}

## Limites

Com apenas dois prompts, sinal e direção podem variar; exigimos apenas endpoints finitos e preservamos os valores observados. O objetivo é demonstrar a semântica do algoritmo em arestas pequenas para a RTX 3050, não reproduzir numericamente todas as figuras do artigo. O efeito histórico também pode passar por MLPs recomputados, exatamente a ressalva do Appendix B.